# Notebook 3 — Boosting, XGBoost, LightGBM & Stacking

**Series:** Ensemble Learning · Notebook 3 of 3  
**Theory depth:** [boosting_and_stacking.md](boosting_and_stacking.md)  
**What you build here:** AdaBoost from scratch, GradientBoosting, HistGradientBoosting, XGBoost, LightGBM, OOF stacking

---

## Section 0 — Setup & Version Check

In [ ]:
# ── Imports and library version verification ─────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import sklearn
from sklearn.datasets import make_classification, load_breast_cancer
from sklearn.model_selection import (train_test_split, cross_val_score,
                                      StratifiedKFold, GridSearchCV)
from sklearn.ensemble import (AdaBoostClassifier, GradientBoostingClassifier,
                               HistGradientBoostingClassifier,
                               RandomForestClassifier, StackingClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, log_loss
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Optional: XGBoost and LightGBM — graceful fallback if not installed
try:
    import xgboost as xgb
    print(f'XGBoost version: {xgb.__version__}')
    XGB_AVAILABLE = True
except ImportError:
    print('XGBoost not installed — skipping XGB cells. Install: pip install xgboost')
    XGB_AVAILABLE = False

try:
    import lightgbm as lgb
    print(f'LightGBM version: {lgb.__version__}')
    LGB_AVAILABLE = True
except ImportError:
    print('LightGBM not installed — skipping LGB cells. Install: pip install lightgbm')
    LGB_AVAILABLE = False

assert sklearn.__version__ >= '1.4', f'Need sklearn >= 1.4, got {sklearn.__version__}'
print(f'sklearn version: {sklearn.__version__}')
np.random.seed(42)

## Section 1 — Quick Theory Recap

Boosting trains models **sequentially**. Each new model corrects the errors of all previous ones.  
AdaBoost: reweights training samples — misclassified examples get higher weight next round.  
Gradient Boosting: fits new trees to **pseudo-residuals** (negative gradient of loss). Works for any differentiable loss.  
XGBoost/LightGBM/CatBoost: high-performance GB implementations with regularization, GPU support, and algorithmic improvements.  
→ Full theory in [boosting_and_stacking.md](boosting_and_stacking.md)

## Section 2 — Data Preparation

In [ ]:
# ── Load and split both datasets ─────────────────────────────────────────────
X_toy, y_toy = make_classification(
    n_samples=2000, n_features=20, n_informative=10,
    n_redundant=4, n_clusters_per_class=2, random_state=42
)
X_toy_tr, X_toy_te, y_toy_tr, y_toy_te = train_test_split(
    X_toy, y_toy, test_size=0.2, stratify=y_toy, random_state=42
)

cancer = load_breast_cancer()
X_can, y_can = cancer.data, cancer.target
feat_names = cancer.feature_names
X_tr, X_te, y_tr, y_te = train_test_split(
    X_can, y_can, test_size=0.2, stratify=y_can, random_state=42
)
# Validation split (from train) for early stopping experiments
X_subtrain, X_val, y_subtrain, y_val = train_test_split(
    X_tr, y_tr, test_size=0.15, random_state=42
)
print(f'Cancer: Train={X_tr.shape}, Val={X_val.shape}, Test={X_te.shape}')

## Section 3 — Model Implementation

### 3a. AdaBoost — The Intuition Verified

In [ ]:
# ── AdaBoost weight update — trace through one iteration manually ─────────────
# This shows exactly what sklearn's AdaBoostClassifier does under the hood
def adaboost_step(X, y, sample_weights, stump):
    """One AdaBoost iteration: fit stump, compute alpha, update weights."""
    stump.fit(X, y, sample_weight=sample_weights)
    preds = stump.predict(X)

    # Weighted error rate: fraction of misclassified examples by weight
    incorrect = (preds != y).astype(float)
    err = np.dot(sample_weights, incorrect) / np.sum(sample_weights)
    err = np.clip(err, 1e-10, 1 - 1e-10)  # avoid log(0)

    # Alpha: model weight — 0 if random, inf if perfect
    alpha = 0.5 * np.log((1 - err) / err)

    # Weight update: upweight misclassified, downweight correct
    new_weights = sample_weights * np.exp(-alpha * (2 * y - 1) * (2 * preds - 1))
    new_weights /= new_weights.sum()  # normalize to sum=1

    return alpha, new_weights, err

# Trace one iteration on a small subset to make outputs readable
n_trace = 100
X_trace, y_trace = X_toy_tr[:n_trace], y_toy_tr[:n_trace]
weights = np.ones(n_trace) / n_trace  # uniform initial weights

for i in range(3):
    stump = DecisionTreeClassifier(max_depth=1, random_state=42)
    alpha, weights, err = adaboost_step(X_trace, y_trace, weights, stump)
    print(f'Round {i+1}: error={err:.4f}, alpha={alpha:.4f}, max_weight={weights.max():.4f}')

In [ ]:
# ── sklearn AdaBoostClassifier — track training and test errors per iteration ─
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # stumps are canonical
    n_estimators=300,
    learning_rate=1.0,
    algorithm='SAMME',  # 'SAMME.R' deprecated in sklearn 1.6; use 'SAMME'
    random_state=42
)
ada.fit(X_tr, y_tr)

# staged_predict gives predictions after each round — perfect for learning curves
ada_train_errors = [
    1 - accuracy_score(y_tr, preds)
    for preds in ada.staged_predict(X_tr)
]
ada_test_errors = [
    1 - accuracy_score(y_te, preds)
    for preds in ada.staged_predict(X_te)
]
print(f'AdaBoost final test accuracy: {accuracy_score(y_te, ada.predict(X_te)):.4f}')

In [ ]:
# ── Gradient Boosting: sklearn GradientBoostingClassifier with staged scores ──
gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=3,        # shallow trees — each corrects a small gradient step
    subsample=0.8,      # stochastic gradient boosting: sample 80% of rows per tree
    random_state=42
)
gb.fit(X_tr, y_tr)

gb_train_loss = list(gb.train_score_)          # deviance per round (training)
gb_test_errors = [
    1 - accuracy_score(y_te, preds)
    for preds in gb.staged_predict(X_te)
]
print(f'GradientBoosting final test accuracy: {accuracy_score(y_te, gb.predict(X_te)):.4f}')

In [ ]:
# ── HistGradientBoostingClassifier — sklearn's fast modern implementation ─────
# Use this instead of GradientBoostingClassifier for datasets > 10k rows
hist_gb = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.1,
    max_depth=4,
    l2_regularization=0.1,  # L2 on leaf weights
    early_stopping=True,    # stops when val score stops improving
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=42
)
hist_gb.fit(X_tr, y_tr)
print(f'HistGradientBoosting: {accuracy_score(y_te, hist_gb.predict(X_te)):.4f}')
print(f'Stopped at iteration: {hist_gb.n_iter_}')

In [ ]:
# ── XGBoost with early stopping ───────────────────────────────────────────────
if XGB_AVAILABLE:
    xgb_model = xgb.XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,   # fraction of features per tree (like RF's max_features)
        reg_alpha=0.1,          # L1 regularization on leaf weights
        reg_lambda=1.0,         # L2 regularization on leaf weights
        eval_metric='logloss',
        early_stopping_rounds=30,
        random_state=42,
        use_label_encoder=False
    )
    xgb_model.fit(
        X_subtrain, y_subtrain,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    xgb_acc = accuracy_score(y_te, xgb_model.predict(X_te))
    print(f'XGBoost Test Accuracy: {xgb_acc:.4f}')
    print(f'Best iteration (early stopping): {xgb_model.best_iteration}')
else:
    print('Skipping XGBoost — not installed')

In [ ]:
# ── LightGBM with leaf-wise growth and early stopping ────────────────────────
if LGB_AVAILABLE:
    lgb_model = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,          # PRIMARY complexity control in LightGBM (not max_depth)
        feature_fraction=0.8,  # colsample_bytree equivalent
        bagging_fraction=0.8,  # row sampling
        bagging_freq=5,        # perform bagging every 5 rounds
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42,
        n_jobs=-1,
        verbose=-1             # suppress verbose output
    )
    callbacks = [lgb.early_stopping(30, verbose=False), lgb.log_evaluation(period=-1)]
    lgb_model.fit(
        X_subtrain, y_subtrain,
        eval_set=[(X_val, y_val)],
        callbacks=callbacks
    )
    lgb_acc = accuracy_score(y_te, lgb_model.predict(X_te))
    print(f'LightGBM Test Accuracy: {lgb_acc:.4f}')
    print(f'Best iteration: {lgb_model.best_iteration_}')
else:
    print('Skipping LightGBM — not installed')

## Section 4 — Stacking with Out-Of-Fold Predictions

The key to correct stacking: **each training example's prediction comes from a model that never saw it** (OOF = Out-Of-Fold).

In [ ]:
# ── OOF stacking from scratch — shows exactly what sklearn StackingClassifier does ──
from sklearn.base import clone

def get_oof_predictions(model, X, y, n_splits=5):
    """Generate out-of-fold predictions. Each sample predicted by a model NOT trained on it."""
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(y))
    oof_proba = np.zeros(len(y))  # probabilities for soft stacking

    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        # Clone to get a fresh untrained model for each fold
        fold_model = clone(model)
        fold_model.fit(X[train_idx], y[train_idx])
        oof_preds[val_idx] = fold_model.predict(X[val_idx])
        oof_proba[val_idx] = fold_model.predict_proba(X[val_idx])[:, 1]

    return oof_preds, oof_proba

# Base models for stacking
base_lr = LogisticRegression(max_iter=300, C=1.0, random_state=42)
base_rf = RandomForestClassifier(n_estimators=100, max_features='sqrt', random_state=42, n_jobs=-1)
base_dt = DecisionTreeClassifier(max_depth=5, random_state=42)

print('Generating OOF predictions for 3 base models (this takes ~10 seconds)...')
_, lr_oof = get_oof_predictions(base_lr, X_tr, y_tr)
_, rf_oof = get_oof_predictions(base_rf, X_tr, y_tr)
_, dt_oof = get_oof_predictions(base_dt, X_tr, y_tr)

# Stack OOF predictions as meta-features
meta_X_train = np.column_stack([lr_oof, rf_oof, dt_oof])
print(f'Meta-feature matrix shape: {meta_X_train.shape}  (one row per training sample, one col per base model)')

In [ ]:
# ── Train meta-learner on OOF predictions ────────────────────────────────────
# Simple meta-learner: logistic regression — complexity here risks overfitting OOF predictions
meta_learner = LogisticRegression(C=1.0, random_state=42)

# At test time: each base model trains on FULL training data, then predicts on test
base_lr.fit(X_tr, y_tr)
base_rf.fit(X_tr, y_tr)
base_dt.fit(X_tr, y_tr)

meta_X_test = np.column_stack([
    base_lr.predict_proba(X_te)[:, 1],
    base_rf.predict_proba(X_te)[:, 1],
    base_dt.predict_proba(X_te)[:, 1]
])

# Fit meta-learner on OOF, evaluate on test
meta_learner.fit(meta_X_train, y_tr)
stack_preds = meta_learner.predict(meta_X_test)
print(f'OOF Stacking Test Accuracy: {accuracy_score(y_te, stack_preds):.4f}')
print(f'Meta-learner coefficients: {meta_learner.coef_[0].round(3)}  ← learned model weights')

In [ ]:
# ── sklearn StackingClassifier: same idea, cleaner API ───────────────────────
sklearn_stacker = StackingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=300, random_state=42)),
        ('rf', RandomForestClassifier(n_estimators=100, max_features='sqrt', random_state=42, n_jobs=-1)),
        ('dt', DecisionTreeClassifier(max_depth=5, random_state=42))
    ],
    final_estimator=LogisticRegression(C=1.0, random_state=42),  # meta-learner
    cv=5,          # 5-fold for OOF predictions (same as our manual implementation)
    stack_method='predict_proba',  # use probabilities, not hard predictions
    passthrough=False,  # set True to also include original features in meta-input
    n_jobs=-1
)
sklearn_stacker.fit(X_tr, y_tr)
print(f'sklearn StackingClassifier accuracy: {accuracy_score(y_te, sklearn_stacker.predict(X_te)):.4f}')

## Section 5 — Visualizations

In [ ]:
# ── Learning curves: AdaBoost and GradientBoosting error vs iterations ────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AdaBoost learning curve
ax = axes[0]
rounds = np.arange(1, len(ada_train_errors) + 1)
ax.plot(rounds, ada_train_errors, label='Train Error', color='#2196F3')
ax.plot(rounds, ada_test_errors, label='Test Error', color='#F44336')
ax.set_xlabel('Number of Estimators')
ax.set_ylabel('Error Rate')
ax.set_title('AdaBoost Learning Curve')
ax.legend()
ax.grid(alpha=0.3)

# GradientBoosting learning curve
ax2 = axes[1]
rounds_gb = np.arange(1, len(gb_test_errors) + 1)
ax2.plot(rounds_gb, gb_train_loss, label='Train Loss (deviance)', color='#2196F3')
ax2.plot(rounds_gb, gb_test_errors, label='Test Error Rate', color='#F44336')
ax2.set_xlabel('Number of Trees')
ax2.set_ylabel('Loss / Error')
ax2.set_title('GradientBoosting Learning Curve')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('boosting_learning_curves.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Effect of learning rate + n_estimators interaction ───────────────────────
lr_values = [0.5, 0.1, 0.05, 0.01]
n_est_range = [10, 25, 50, 100, 200, 400]

fig, ax = plt.subplots(figsize=(10, 5))
colors_lr = ['#E53935', '#FB8C00', '#43A047', '#1E88E5']

for lr_val, color in zip(lr_values, colors_lr):
    test_accs = []
    for n in n_est_range:
        m = HistGradientBoostingClassifier(
            max_iter=n, learning_rate=lr_val, random_state=42
        )
        m.fit(X_tr, y_tr)
        test_accs.append(accuracy_score(y_te, m.predict(X_te)))
    ax.plot(n_est_range, test_accs, 'o-', label=f'lr={lr_val}', color=color)

ax.set_xlabel('Number of Boosting Rounds')
ax.set_ylabel('Test Accuracy')
ax.set_title('Learning Rate × n_estimators Interaction\n(low lr needs more rounds but generalizes better)')
ax.legend(title='Learning Rate')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('lr_n_estimators_interaction.png', dpi=120, bbox_inches='tight')
plt.show()

## Section 6 — Comparison Table

In [ ]:
# ── Master comparison: all ensemble methods on Breast Cancer ─────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model_zoo = [
    ('Decision Tree (baseline)',   DecisionTreeClassifier(random_state=42)),
    ('Random Forest (200 trees)',  RandomForestClassifier(n_estimators=200, max_features='sqrt', random_state=42, n_jobs=-1)),
    ('AdaBoost (300 stumps)',      AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=300, algorithm='SAMME', random_state=42)),
    ('GradientBoosting (200)',     GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42)),
    ('HistGradientBoosting',       HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=42)),
    ('Stacking (LR meta-learner)', sklearn_stacker),
]

if XGB_AVAILABLE:
    model_zoo.append(('XGBoost', xgb.XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=42, use_label_encoder=False, eval_metric='logloss')))
if LGB_AVAILABLE:
    model_zoo.append(('LightGBM', lgb.LGBMClassifier(n_estimators=200, learning_rate=0.1, num_leaves=31, random_state=42, n_jobs=-1, verbose=-1)))

rows = []
for name, model in model_zoo:
    model.fit(X_tr, y_tr)
    test_acc = accuracy_score(y_te, model.predict(X_te))
    cv_scores = cross_val_score(model, X_can, y_can, cv=cv)
    rows.append({
        'Model': name,
        'Test Acc': round(test_acc, 4),
        'CV Mean': round(cv_scores.mean(), 4),
        'CV Std': round(cv_scores.std(), 4)
    })

df = pd.DataFrame(rows).sort_values('CV Mean', ascending=False).reset_index(drop=True)
df.style.highlight_max(subset=['Test Acc', 'CV Mean'], color='lightgreen') \
        .highlight_min(subset=['CV Std'], color='lightyellow') \
        .set_caption('Complete Ensemble Comparison — Breast Cancer Wisconsin')

## Section 7 — Key Takeaways

- **Gradient Boosting = gradient descent in function space.** Each tree fits pseudo-residuals (negative gradient of loss). This framing explains *why* learning rate matters: it's step size in gradient descent.
- **Learning rate and n_estimators are coupled.** Lower learning rate requires more trees. Always tune them together, using early stopping to find the optimal n_estimators automatically.
- **HistGradientBoostingClassifier** is sklearn's fast modern implementation for large datasets — use it instead of GradientBoostingClassifier for datasets with more than ~10k rows.
- **OOF predictions are mandatory in stacking.** Training base models on full training data and then generating predictions for the meta-learner causes target leakage — the meta-learner over-trusts overfitted in-sample predictions.
- **Simple meta-learner wins in stacking.** Logistic regression as the meta-learner is the standard choice — the base model OOF probabilities are already rich features; additional complexity at the meta level risks overfitting.